# Sentinel-2 Water Quality Processing Workflow

This notebook demonstrates the complete workflow for processing Sentinel-2 data to extract water quality parameters.

## Overview

The workflow includes:
1. **Data Download** - Download Sentinel-2 L1C data from Copernicus
2. **Preprocessing** - Resample, subset, and reproject data
3. **Water Quality Processing** - Apply C2RCC and calculate CDOM
4. **Visualization** - Generate plots for CHL, TSM, CDOM, and True Color

## Directory Structure

```
Sentinel2_WQ/
├── 01_scripts/                 # Processing scripts
├── 02_config/                  # Configuration files
├── 03_raw_data/               # Downloaded raw data
├── 04_processed_data/         # Intermediate processed data
├── 05_final_products/         # Final output products
├── 06_logs/                   # Processing logs
└── 07_documentation/          # Documentation
```

## Setup and Configuration

In [ ]:
import os
import sys
from pathlib import Path
import subprocess

# Set base directory
base_dir = Path("d:/Sentinel2_WQ")
os.chdir(base_dir)

# Add scripts directory to path
sys.path.append(str(base_dir / "01_scripts"))

print(f"Working directory: {os.getcwd()}")
print(f"Base directory: {base_dir}")

## Check Dependencies

In [ ]:
# Check if requirements are installed
try:
    import numpy as np
    import matplotlib.pyplot as plt
    import xarray as xr
    import cartopy.crs as ccrs
    import cmocean
    import geopandas as gpd
    import pandas as pd
    import requests
    
    print("✓ All required Python packages are available")
except ImportError as e:
    print(f"✗ Missing package: {e}")
    print("Please install requirements: pip install -r requirements.txt")

## Check SNAP Installation

In [ ]:
# Check SNAP GPT installation
try:
    result = subprocess.run(['gpt', '--help'], capture_output=True, text=True)
    if result.returncode == 0:
        print("✓ SNAP GPT is installed and working")
    else:
        print("✗ SNAP GPT is not working properly")
except FileNotFoundError:
    print("✗ SNAP GPT not found. Please install SNAP and add it to PATH")

## View Current Processing Statistics

In [ ]:
# Import utility functions
import utils

# Print current processing statistics
utils.print_processing_statistics(base_dir)

## Option 1: Run Complete Workflow

In [ ]:
# Run the complete workflow
import subprocess
import sys

# Run master script
result = subprocess.run([
    sys.executable, 
    "run_workflow.py", 
    "--action", "full",
    "--config", "02_config/parameters.yaml"
], capture_output=True, text=True)

print("STDOUT:")
print(result.stdout)
print("\nSTDERR:")
print(result.stderr)
print(f"\nReturn code: {result.returncode}")

## Option 2: Run Individual Steps

### Step 1: Download Data

In [ ]:
# Download Sentinel-2 data
from download import Sentinel2Downloader
from datetime import datetime

# Initialize downloader
config_path = base_dir / "02_config/parameters.yaml"
downloader = Sentinel2Downloader(config_path)

# Set date range
start_date = datetime(2025, 5, 1)
end_date = datetime(2025, 6, 30)

# Download products
downloader.download_products(start_date, end_date)

### Step 2: Process Data

In [ ]:
# Process the data
from process_pipeline import WaterQualityProcessor

# Initialize processor
processor = WaterQualityProcessor(config_path)

# Run full pipeline
success = processor.run_full_pipeline()

if success:
    print("Processing completed successfully!")
else:
    print("Processing failed!")

### Step 3: Generate Plots

In [ ]:
# Generate plots
from plotting import WaterQualityPlotter

# Initialize plotter
plotter = WaterQualityPlotter(config_path)

# Generate all plots
success = plotter.generate_all_plots()

if success:
    print("Plot generation completed successfully!")
else:
    print("Plot generation failed!")

## View Results

In [ ]:
# Display sample results
import matplotlib.pyplot as plt
from pathlib import Path

# Check for generated plots
products_dir = base_dir / "05_final_products"

fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('Sentinel-2 Water Quality Products', fontsize=16)

# Plot types and their directories
plot_types = {
    'Chlorophyll-a': 'chl',
    'Total Suspended Matter': 'tsm', 
    'CDOM': 'cdom',
    'True Color': 'true_color'
}

for i, (title, dir_name) in enumerate(plot_types.items()):
    ax = axes[i // 2, i % 2]
    
    plot_dir = products_dir / dir_name
    if plot_dir.exists():
        png_files = list(plot_dir.glob('*.png'))
        if png_files:
            # Load and display the first available image
            img_path = png_files[0]
            img = plt.imread(img_path)
            ax.imshow(img)
            ax.set_title(f'{title}\n{img_path.name}')
        else:
            ax.text(0.5, 0.5, f'No {title} plots found', 
                   ha='center', va='center', transform=ax.transAxes)
            ax.set_title(title)
    else:
        ax.text(0.5, 0.5, f'Directory not found:\n{plot_dir}', 
               ha='center', va='center', transform=ax.transAxes)
        ax.set_title(title)
    
    ax.axis('off')

plt.tight_layout()
plt.show()

# Print final statistics
print("\nFinal Processing Statistics:")
utils.print_processing_statistics(base_dir)

## Workflow Summary

This notebook demonstrates the complete Sentinel-2 water quality processing workflow:

1. **Setup**: Check dependencies and configuration
2. **Download**: Fetch Sentinel-2 L1C data from Copernicus
3. **Process**: 
   - Resample and subset data
   - Reproject to WGS84
   - Generate true color images
   - Apply C2RCC atmospheric correction
   - Calculate CDOM using band math
4. **Visualize**: Generate final plots for all water quality parameters

The optimized workflow provides:
- **Organized structure** with clear separation of scripts, data, and outputs
- **Automated processing** with error handling and logging
- **Flexible configuration** using YAML files
- **Comprehensive logging** for troubleshooting
- **Scalable design** for processing multiple dates and areas

## Next Steps

1. **Customize configuration** in `02_config/parameters.yaml`
2. **Add your credentials** for Copernicus data access
3. **Adjust processing parameters** for your specific study area
4. **Run the workflow** using the batch script or individual steps
5. **Analyze results** in the `05_final_products` directory